# Airbnb Paris – Semantisch (no PCA)
- Freitexte per `all-mpnet-base-v2` (768d) eingebettet
- Spaltennamen-Embedding addiert (Positionskodierung), danach per-Vektor-LayerNorm

In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

/home/debian/TFM_master_thesis/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Cleaned+Text laden & Modell initialisieren

In [2]:
df = pd.read_csv("../../data/preprocessed/cleaned_text_airbnb_paris.csv")
text_cols = ["name", "description", "neighborhood_overview", "host_about"]

model = SentenceTransformer("all-mpnet-base-v2")
name_emb = {c: model.encode(c) for c in text_cols}  # cached, constant per column

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4004.18it/s]


## Zellen einbetten, Spaltennamen addieren, LayerNorm

In [3]:
parts = []
for c in text_cols:
    e = model.encode(df[c].fillna("").tolist(), batch_size=64, show_progress_bar=False) + name_emb[c]
    e = (e - e.mean(axis=1, keepdims=True)) / (e.std(axis=1, keepdims=True) + 1e-6)
    cols = [f"{c}_emb_{i}" for i in range(e.shape[1])]
    parts.append(pd.DataFrame(e, columns=cols, index=df.index))

## Rohtexte ersetzen & speichern

In [4]:
out = pd.concat([df.drop(columns=text_cols)] + parts, axis=1)
out.to_csv("../../data/preprocessed/semantic_airbnb_paris.csv", index=False)

## Verifikation

In [5]:
print("shape", out.shape)
print("embedding columns:", len([c for c in out.columns if "_emb_" in c]))
assert out.isna().sum().sum() == 0

shape (18350, 3128)
embedding columns: 3072
